# Documentación del Dataset de Golpes Revisados

Este notebook explica qué representa cada columna del dataset `golpes_revisados.csv`,
utilizado como conjunto de etiquetas verificadas manualmente para el análisis de golpeos en pádel.

El dataset fue generado mediante:

1. Selección de frames candidatos a golpeo a partir de `dataset_padel.csv`.
2. Revisión manual frame a frame en video para confirmar que el jugador efectivamente impacta la pelota con la raqueta.
3. Registro del jugador que golpea y el cluster de K-means al que pertenece ese frame.

---

## Propósito

Este dataset **no** se generó automáticamente con reglas angulares.
Cada fila fue verificada a ojo en el video correspondiente.
Eso lo convierte en la fuente de verdad (ground truth) del proyecto,
libre del problema de etiquetado circular que afecta a las columnas `Y_golpe` de `dataset_padel.csv`.

## Estructura

El dataset tiene **7 columnas** y **81 filas** (una por golpeo confirmado).


## Columna 1 — `cancha`

In [1]:
columna_cancha = {
    "tipo": "Categórica (string)",
    "descripcion": "Identificador de la cancha física donde se jugó el partido.",
    "valores_posibles": ["Cancha_1"],
    "nota": "Todos los golpeos revisados pertenecen a Cancha_1. "
            "Cancha_2 no tiene golpeos revisados manualmente en este dataset."
}

for k, v in columna_cancha.items():
    print(f"{k}: {v}")


tipo: Categórica (string)
descripcion: Identificador de la cancha física donde se jugó el partido.
valores_posibles: ['Cancha_1']
nota: Todos los golpeos revisados pertenecen a Cancha_1. Cancha_2 no tiene golpeos revisados manualmente en este dataset.


## Columna 2 — `partido`

In [2]:
import pandas as pd

df = pd.read_csv("datasets/golpes_revisados.csv", sep="\t")

columna_partido = {
    "tipo": "Categórica (string)",
    "descripcion": "Identificador del partido dentro de la cancha. "
                  "Corresponde al mismo identificador que en dataset_padel.csv.",
    "ejemplo": "Partido_15, Partido_3, Partido_37"
}

for k, v in columna_partido.items():
    print(f"{k}: {v}")

print()
print("Partidos con golpeos revisados:")
print(sorted(df["partido"].unique()))
print(f"Total: {df['partido'].nunique()} partidos distintos")


tipo: Categórica (string)
descripcion: Identificador del partido dentro de la cancha. Corresponde al mismo identificador que en dataset_padel.csv.
ejemplo: Partido_15, Partido_3, Partido_37

Partidos con golpeos revisados:
['Partido_1', 'Partido_15', 'Partido_17', 'Partido_19', 'Partido_21', 'Partido_3', 'Partido_31', 'Partido_33', 'Partido_35', 'Partido_37', 'Partido_5', 'Partido_9']
Total: 12 partidos distintos


## Columna 3 — `punto`

In [3]:
columna_punto = {
    "tipo": "Categórica (string)",
    "descripcion": "Identificador del punto dentro del partido. "
                  "Un punto es un video desde el saque hasta el fin del rally. "
                  "Corresponde al mismo identificador que en dataset_padel.csv.",
    "ejemplo": "Punto_6, Punto_7, Punto__11"
}

for k, v in columna_punto.items():
    print(f"{k}: {v}")

print()
# Detectar anomalías en el nombre del punto
puntos_doble_guion = [p for p in df["punto"].unique() if "__" in p]
if puntos_doble_guion:
    print("NOTA — Puntos con doble guion bajo (anomalía de nomenclatura):")
    print(puntos_doble_guion)
    print("Estos puntos tienen doble guion bajo (Punto__11 en lugar de Punto_11)."
          " Son correctos como identificadores pero difieren del patrón habitual.")


tipo: Categórica (string)
descripcion: Identificador del punto dentro del partido. Un punto es un video desde el saque hasta el fin del rally. Corresponde al mismo identificador que en dataset_padel.csv.
ejemplo: Punto_6, Punto_7, Punto__11

NOTA — Puntos con doble guion bajo (anomalía de nomenclatura):
['Punto__11', 'Punto__12', 'Punto__10']
Estos puntos tienen doble guion bajo (Punto__11 en lugar de Punto_11). Son correctos como identificadores pero difieren del patrón habitual.


## Columna 4 — `frame`

In [4]:
columna_frame = {
    "tipo": "Entero (int)",
    "descripcion": "Número del frame dentro del video del punto. "
                  "Los frames van de 3 en 3 porque el pipeline procesó 1 de cada 3 frames.",
    "ejemplo": "324, 339, 951"
}

for k, v in columna_frame.items():
    print(f"{k}: {v}")

print()
print(f"Rango de frames: {df['frame'].min()} — {df['frame'].max()}")
print()
print("Los frames son múltiplos de 3 por el submuestreo del pipeline:")
es_multiplo_3 = (df["frame"] % 3 == 0).sum()
print(f"  {es_multiplo_3} de {len(df)} frames son múltiplos de 3")


tipo: Entero (int)
descripcion: Número del frame dentro del video del punto. Los frames van de 3 en 3 porque el pipeline procesó 1 de cada 3 frames.
ejemplo: 324, 339, 951

Rango de frames: 12 — 1725

Los frames son múltiplos de 3 por el submuestreo del pipeline:
  81 de 81 frames son múltiplos de 3


## Columna 5 — `jugador`

In [5]:
columna_jugador = {
    "tipo": "Categórica (string)",
    "descripcion": "Identificador del jugador que ejecuta el golpeo en ese frame. "
                  "Un partido de pádel tiene 4 jugadores: j1, j2, j3, j4. "
                  "Esta columna indica cuál de los 4 está golpeando la pelota.",
    "valores_posibles": ["j1", "j2", "j3", "j4"],
    "nota": "Un mismo frame puede tener un único golpeador. "
            "Solo se registra el jugador que impacta la pelota."
}

for k, v in columna_jugador.items():
    print(f"{k}: {v}")

print()
print("Distribución de golpeos por jugador:")
print(df["jugador"].value_counts().sort_index())


tipo: Categórica (string)
descripcion: Identificador del jugador que ejecuta el golpeo en ese frame. Un partido de pádel tiene 4 jugadores: j1, j2, j3, j4. Esta columna indica cuál de los 4 está golpeando la pelota.
valores_posibles: ['j1', 'j2', 'j3', 'j4']
nota: Un mismo frame puede tener un único golpeador. Solo se registra el jugador que impacta la pelota.

Distribución de golpeos por jugador:
jugador
j1    20
j2    20
j3    21
j4    20
Name: count, dtype: int64


## Columna 6 — `golpeo`

In [6]:
columna_golpeo = {
    "tipo": "Entero binario (int)",
    "descripcion": "Indica si en ese frame ocurre un golpeo real del jugador indicado. "
                  "Verificado manualmente en el video correspondiente.",
    "valor_unico": 1,
    "interpretacion": "1 = golpeo confirmado visualmente. "
                     "Este dataset solo contiene golpeos confirmados, "
                     "por eso todos los valores son 1."
}

for k, v in columna_golpeo.items():
    print(f"{k}: {v}")

print()
print("Distribución del valor golpeo:")
print(df["golpeo"].value_counts())
print()
print("Por qué solo hay 1s:")
print("  Este dataset es el resultado de revisar manualmente frames sospechosos")
print("  y confirmar cuáles realmente son golpes. Los no-golpes no se registran aquí.")
print("  Los no-golpes corresponden al resto del dataset_padel.csv.")


tipo: Entero binario (int)
descripcion: Indica si en ese frame ocurre un golpeo real del jugador indicado. Verificado manualmente en el video correspondiente.
valor_unico: 1
interpretacion: 1 = golpeo confirmado visualmente. Este dataset solo contiene golpeos confirmados, por eso todos los valores son 1.

Distribución del valor golpeo:
golpeo
1    81
Name: count, dtype: int64

Por qué solo hay 1s:
  Este dataset es el resultado de revisar manualmente frames sospechosos
  y confirmar cuáles realmente son golpes. Los no-golpes no se registran aquí.
  Los no-golpes corresponden al resto del dataset_padel.csv.


## Columna 7 — `cluster`

In [7]:
columna_cluster = {
    "tipo": "Float (puede ser NaN)",
    "descripcion": "Cluster de K-means al que pertenece este frame-jugador, "
                  "según el modelo entrenado en kmeans/kmean.ipynb. "
                  "El clustering se realizó sobre los 5 ángulos biomecánicos del jugador "
                  "(angulo_codo_d, angulo_hombro_d, angulo_inclinacion_raqueta_d, "
                  "angulo_raqueta_hombro_d, angulo_muneca_cabeza_d).",
    "valores_posibles": [0, 1, 2, "NaN"],
    "por_que_float": "Es float y no int porque Python/pandas usa NaN para valores "
                    "faltantes, y NaN solo existe en tipo float64."
}

for k, v in columna_cluster.items():
    print(f"{k}: {v}")

print()
print("Distribución de clusters:")
print(df["cluster"].value_counts(dropna=False).sort_index())
print()
print(f"Filas con NaN en cluster: {df['cluster'].isna().sum()}")
print()
print("Por qué hay NaN:")
print("  Los NaN corresponden a jugadores (principalmente j3 y j4) que no fueron")
print("  detectados correctamente por el modelo de pose estimation en ese frame.")
print("  Sin detección no hay ángulos, y sin ángulos no hay cluster asignable.")


tipo: Float (puede ser NaN)
descripcion: Cluster de K-means al que pertenece este frame-jugador, según el modelo entrenado en kmeans/kmean.ipynb. El clustering se realizó sobre los 5 ángulos biomecánicos del jugador (angulo_codo_d, angulo_hombro_d, angulo_inclinacion_raqueta_d, angulo_raqueta_hombro_d, angulo_muneca_cabeza_d).
valores_posibles: [0, 1, 2, 'NaN']
por_que_float: Es float y no int porque Python/pandas usa NaN para valores faltantes, y NaN solo existe en tipo float64.

Distribución de clusters:
cluster
0.0    23
1.0    32
2.0     2
NaN    24
Name: count, dtype: int64

Filas con NaN en cluster: 24

Por qué hay NaN:
  Los NaN corresponden a jugadores (principalmente j3 y j4) que no fueron
  detectados correctamente por el modelo de pose estimation en ese frame.
  Sin detección no hay ángulos, y sin ángulos no hay cluster asignable.


## Por qué hay NaN en `cluster`

El K-means se aplicó en formato per-jugador: solo se procesaron filas
donde los **5 ángulos biomecánicos** del jugador estaban disponibles.

Si el modelo de pose estimation no detectó al jugador en ese frame
(por oclusión, salida del encuadre, etc.), los ángulos son NaN
y ese frame-jugador queda fuera del clustering.

Esto afecta principalmente a **j3 y j4**, que están más alejados de la cámara:


In [8]:
nan_por_jugador = df[df["cluster"].isna()]["jugador"].value_counts()
print("NaN en cluster por jugador:")
print(nan_por_jugador)
print()
print("Golpeos con cluster asignado vs sin asignar:")
print(f"  Con cluster : {df['cluster'].notna().sum()} de {len(df)}")
print(f"  Sin cluster : {df['cluster'].isna().sum()} de {len(df)}")


NaN en cluster por jugador:
jugador
j4    16
j3     8
Name: count, dtype: int64

Golpeos con cluster asignado vs sin asignar:
  Con cluster : 57 de 81
  Sin cluster : 24 de 81


## Relación con `dataset_padel.csv`

`golpes_revisados.csv` es un **subconjunto indexado** de `dataset_padel.csv`.

Cada fila de `golpes_revisados.csv` puede unirse con `dataset_padel.csv`
usando las columnas `(cancha, partido, punto, frame)` para obtener
todas las features del frame: ángulos, posiciones, confianzas.

```text
golpes_revisados.csv
  cancha + partido + punto + frame  ──►  dataset_padel.csv  (features completas)
  jugador                           ──►  columnas j1_* / j2_* / j3_* / j4_*
```

El campo `jugador` indica qué bloque de columnas usar en `dataset_padel.csv`:
si `jugador = "j2"`, las features relevantes son `j2_angulo_codo_d`, `j2_confianza`, etc.


In [9]:
# Verificar que todos los golpes existen en dataset_padel.csv
df_padel = pd.read_csv("datasets/dataset_padel.csv")

merged = df.merge(
    df_padel[["cancha", "partido", "punto", "frame"]],
    on=["cancha", "partido", "punto", "frame"],
    how="left",
    indicator=True
)

print("Verificación: golpeos revisados que existen en dataset_padel.csv")
print(merged["_merge"].value_counts())
print()
print("Todos los 81 golpeos tienen su frame correspondiente en dataset_padel.csv.")


Verificación: golpeos revisados que existen en dataset_padel.csv
_merge
both          81
left_only      0
right_only     0
Name: count, dtype: int64

Todos los 81 golpeos tienen su frame correspondiente en dataset_padel.csv.


## Resumen estructural

```text
golpes_revisados.csv
│
├── Identificación del frame
│   ├── cancha   → siempre "Cancha_1"
│   ├── partido  → 12 partidos distintos
│   ├── punto    → punto dentro del partido
│   └── frame    → número de frame (múltiplos de 3)
│
├── jugador  → quién golpea (j1 / j2 / j3 / j4)
│
├── golpeo   → siempre 1 (golpe confirmado manualmente)
│
└── cluster  → cluster K-means del frame-jugador
               (NaN si el jugador no fue detectado)
```

## Estadísticas globales


In [10]:
print("=" * 45)
print("  RESUMEN golpes_revisados.csv")
print("=" * 45)
print(f"  Total filas          : {len(df)}")
print(f"  Cancha única         : {df['cancha'].nunique()} (solo Cancha_1)")
print(f"  Partidos             : {df['partido'].nunique()}")
print(f"  Puntos               : {df['punto'].nunique()}")
print(f"  Jugadores            : {df['jugador'].nunique()} (j1, j2, j3, j4)")
print(f"  Golpeos confirmados  : {df['golpeo'].sum()} (todos)")
print(f"  Con cluster          : {df['cluster'].notna().sum()}")
print(f"  Sin cluster (NaN)    : {df['cluster'].isna().sum()}")
print("=" * 45)
print()
print("Golpeos por jugador:")
print(df["jugador"].value_counts().sort_index().to_string())
print()
print("Golpeos por cluster (excluyendo NaN):")
print(df["cluster"].value_counts().sort_index().to_string())


  RESUMEN golpes_revisados.csv
  Total filas          : 81
  Cancha única         : 1 (solo Cancha_1)
  Partidos             : 12
  Puntos               : 15
  Jugadores            : 4 (j1, j2, j3, j4)
  Golpeos confirmados  : 81 (todos)
  Con cluster          : 57
  Sin cluster (NaN)    : 24

Golpeos por jugador:
jugador
j1    20
j2    20
j3    21
j4    20

Golpeos por cluster (excluyendo NaN):
cluster
0.0    23
1.0    32
2.0     2
